In [ ]:
import pandas as pd
import numpy as np
from tools import sherlock
import calendar

In [ ]:
df = pd.read_excel(r"..\data\raw\taxe_de_sejour_2024.ods", engine='odf', nrows=41)

# Exploration

In [ ]:
sherlock(df)

In [ ]:
df = df.rename(str.lower, axis='columns')
df.columns = df.columns.str.strip().str.replace(' ', '_')
df = df.drop(columns=[
    'unnamed:_0', 'motifs_d\'exonérations_de_la_taxe', 'd._nombre_d\'exonérations',
    'g._tarif_unitaire_de_la_taxe', 'h._montant_de_la_taxe_perçue_exbxg', 'a._montant_de_la_nuitée'
    ])
df = df.rename(columns={
    'date_de_perception_de_la_taxe__(date_du_séjour':'date', 
    'b._nombre_de_nuitées_enregistrées':'nb_nuit',
    'c._nombre_de_personnes_qui_ont_séjournées':'nb_personnes',
    'e._prix_par_nuitée_par_personne_(hébergement_non_classé)_a/c':'prix_nuitée_par_personne'
    })

df['prix_nuitée'] = df['prix_nuitée_par_personne'] * df['nb_personnes']

In [ ]:
sherlock(df)

In [ ]:
df.head()

# Feature

In [ ]:
df['mois'] = df['date'].dt.to_period('M')
df['jours_dispo'] = df.apply(lambda row: calendar.monthrange(row['date'].year, row['date'].month)[1], axis=1)
df.loc[df['mois'] == '2024-07', 'jours_dispo'] = 31 - 16 + 1

df = df.groupby(df['mois']).agg(
    occupation = ('nb_nuit', 'sum'),
    prix_moyen_nuitée = ('prix_nuitée', 'mean'),
    duree_moyenne_sejour = ('nb_nuit', 'mean'),
    nb_sejours = ('date', 'count'),
    jours_dispo_mois = ('jours_dispo', 'max')
).reset_index()

df['taux_occupation'] = df['occupation'] / df['jours_dispo_mois']

In [ ]:
df

### Export

In [ ]:
df.to_csv(r"..\data\processed\tx_occupation.csv", index=False, sep=';', decimal=',', encoding='utf-8-sig')